# Module 3: The Counterfactual You Have to Construct

*Developed by Yin Zhang, PhD, Assistant Professor, Department of Mathematics and Statistics, Washington State University. Part of the Public Safety Statistics Tutorials, developed for WADEPS through CISER.*

---

[Module 2](Module_02_Potential_Outcomes_Without_The_Algebra.ipynb) established
that Y(0) is always missing and always has to be built. This module builds it
three ways on the same agency, gets three different answers, and works out
what to do about that.

**About 25 minutes.**

## 1. Setup

In [ ]:
# Where the data lives.
#   On Google Colab this reads straight from GitHub.
#   Running from inside a local clone of the repository also works.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

GITHUB = "https://raw.githubusercontent.com/YinZhangCISER/Public-Safety-Statistics-Tutorials/main/Data/"
_local = Path("../../../Data")
BASE = f"{_local}/" if _local.exists() else GITHUB

monthly = pd.read_csv(BASE + "agency_monthly.csv")
profile = pd.read_csv(BASE + "agency_profile.csv")

# The five agencies that adopted the de escalation training in July 2023.
TRAINED = ["A001", "A002", "A004", "A007", "A010"]
TRUTH = -12.0                       # the effect built into the data, in percent

f = monthly[monthly["provisional"] == 0].copy()          # drop the unfinished months
f = f[~((f["agency_id"] == "A002") & (f["year_month"] == "2021-06"))]   # documented unrest
f["trained"] = f["agency_id"].isin(TRAINED).astype(int)

# Three periods, not two. The program phased in between July and November 2023.
f["period"] = np.where(f["year_month"] >= "2023-11", "after",
                       np.where(f["year_month"] < "2023-07", "before", "phase"))

NAME = dict(zip(profile["agency_id"], profile["agency_name"]))
COMPARISON = sorted(a for a in f["agency_id"].unique() if a not in TRAINED)


def rate(d):
    """Use of force per 100 arrests, pooled over whatever rows are passed in."""
    return 100 * d["n_uof"].sum() / d["n_arrests"].sum()


def cell_rate(agencies, period):
    return rate(f[f["agency_id"].isin(agencies) & (f["period"] == period)])


print(f"{f['agency_id'].nunique()} agencies, {f['year_month'].nunique()} months")
print(f"trained: {', '.join(NAME[a].split()[0] for a in TRAINED)}")

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf


def own_trend_y0(agency_id):
    """Extrapolate an agency's own pre program trend across the after period."""
    g = f[f["agency_id"] == agency_id]
    pre = g[g["period"] == "before"].copy()
    i1 = pd.PeriodIndex(pre["year_month"], freq="M")
    pre["yr"] = i1.year.values + (i1.month.values - 1) / 12.0
    z = smf.glm("n_uof ~ yr", pre, family=sm.families.Poisson(),
                offset=np.log(pre["n_arrests"])).fit()
    post = g[g["period"] == "after"].copy()
    i2 = pd.PeriodIndex(post["year_month"], freq="M")
    post["yr"] = i2.year.values + (i2.month.values - 1) / 12.0
    return 100 * np.exp(z.params["Intercept"] + z.params["yr"] * post["yr"]).mean()

## 2. Three ways to build Y(0)

| Construction | What it assumes | What it costs |
|---|---|---|
| **The agency's own past, unchanged** | nothing changed over four years | nothing, and it is almost always wrong |
| **The agency's own pre program trend** | whatever was happening kept happening | a few years of history, and no comparison group |
| **The comparison agencies' path** | the two groups would have moved together | a comparison group, and the check in Module 7 |

Only the third uses information from outside the agency. That is its
advantage and the reason it is the default.

In [ ]:
a = "A001"                                   # Stonewick
observed = cell_rate([a], "after")
own_past = cell_rate([a], "before")
own_trend = own_trend_y0(a)
c_mult = cell_rate(COMPARISON, "after") / cell_rate(COMPARISON, "before")
from_comparison = own_past * c_mult

rows = [("its own past, assumed unchanged", own_past),
        ("its own pre program trend, extrapolated", own_trend),
        ("the comparison agencies' path", from_comparison)]
print(f"  {NAME[a]}")
print(f"  what was recorded after the program: {observed:.2f}\n")
print("  counterfactual                            Y(0)    effect")
for lab, y0 in rows:
    print(f"  {lab:40s}  {y0:.2f}   {100 * (observed / y0 - 1):+6.1f}%")
print(f"\n  the truth: {TRUTH:+.1f}%")

Same records, same agency, same after period. **Three counterfactuals, three
answers, and none of them is 12 percent.**

The first is wrong by more than a factor of two, and it is wrong in the
predictable direction: assuming nothing would have changed credits the program
with four years of statewide decline.

The other two land at 8 and 9 percent. They are not right either, and the
reason is not the counterfactual. It is that **one agency does not contain
enough information** to resolve a 12 percent effect, whatever you compare it
against. [Module 14](Module_14_How_Big_An_Effect.ipynb) measures that directly.

## 3. The same three, pooled across four agencies

In [ ]:
keep = [x for x in TRAINED if x != "A007"]
obs = cell_rate(keep, "after")
past = cell_rate(keep, "before")
trend = np.mean([own_trend_y0(x) for x in keep])     # rough: equal weight per agency
comp = past * c_mult

pd.DataFrame([
    {"counterfactual": "its own past, unchanged", "Y(0)": round(past, 2),
     "estimate": f"{100 * (obs / past - 1):+.1f}%"},
    {"counterfactual": "own pre program trend", "Y(0)": round(trend, 2),
     "estimate": f"{100 * (obs / trend - 1):+.1f}%"},
    {"counterfactual": "the comparison agencies", "Y(0)": round(comp, 2),
     "estimate": f"{100 * (obs / comp - 1):+.1f}%"},
    {"counterfactual": "THE TRUTH", "Y(0)": "", "estimate": f"{TRUTH:+.1f}%"},
]).set_index("counterfactual")

Pooling four agencies moves both defensible counterfactuals from 8 and 9
percent up to 13.2 and 12.5, close to the truth and close to each other. The
indefensible one stays where it was, at nearly 30 percent.

**Note which one pooling does not rescue.** More data does not fix a wrong
counterfactual; it just makes a wrong answer more precise.

## 4. When two constructions disagree

Agreement between two counterfactuals is weak evidence for both.
Disagreement is strong evidence that at least one is wrong, and it is worth
finding out which.

In [ ]:
rows = []
for x in keep:
    y1 = cell_rate([x], "after")
    y0c = cell_rate([x], "before") * c_mult
    y0t = own_trend_y0(x)
    rows.append({"agency": NAME[x].split()[0],
                 "from comparison": f"{100 * (y1 / y0c - 1):+.1f}%",
                 "from own trend": f"{100 * (y1 / y0t - 1):+.1f}%",
                 "gap": round(abs(100 * (y1 / y0c - 1) - 100 * (y1 / y0t - 1)), 1)})
pd.DataFrame(rows).set_index("agency").sort_values("gap")

Pinecrest disagrees by nine points, more than twice any other agency.

There is a reason, and it is in the data dictionary rather than the numbers.
**Pinecrest is the campus police agency.** Its calls follow the academic
calendar, peaking in September and collapsing in June and July, while every
other agency in the state peaks in summer. Assuming it would have moved with
the others is exactly the assumption to doubt, and the gap is the data saying
so before anyone looked it up.

## 5. What to report

| Do | Do not |
|---|---|
| Name the counterfactual in one sentence | say "compared to before" and leave it |
| Give the estimate under the most credible alternative too | report only the one you preferred |
| Explain any agency where the two disagree sharply | drop it quietly |
| Say the counterfactual cannot be verified | imply the comparison group proves anything |

A report that gives 12.5 percent from the comparison group and notes that the
own trend construction gives a similar figure at three of four agencies is
far more convincing than one that gives a single number with no alternative.

## Exercise

Build a fourth counterfactual: instead of all seven comparison agencies, use
only the ones most similar in size to each trained agency.

In [ ]:
# Fill in the blank, then run.
RUN = None          # try True

if RUN:
    size = profile.set_index("agency_id")["sworn_officers"]
    rows = []
    for x in keep:
        near = sorted(COMPARISON, key=lambda c: abs(np.log(size[c] / size[x])))[:3]
        m_near = cell_rate(near, "after") / cell_rate(near, "before")
        y1 = cell_rate([x], "after")
        y0_all = cell_rate([x], "before") * c_mult
        y0_near = cell_rate([x], "before") * m_near
        rows.append({"agency": f"{NAME[x].split()[0]} ({size[x]:.0f})",
                     "three nearest in size":
                         ", ".join(f"{NAME[c].split()[0]}" for c in near),
                     "all seven": f"{100 * (y1 / y0_all - 1):+.1f}%",
                     "nearest three": f"{100 * (y1 / y0_near - 1):+.1f}%"})
    print(f"  the truth is {TRUTH:+.1f} percent\n")
    display(pd.DataFrame(rows).set_index("agency"))
else:
    print("Set RUN above, then run this cell again.")

<details>
<summary><b>Solution</b></summary>

```python
RUN = True
```

Matching on size makes the estimates **worse and more variable**, not better.

The reason is the one from [Module 1](Module_01_From_It_Went_Down_To_The_Program_Did_It.ipynb):
each agency is now compared against three agencies instead of seven, so three
sets of accidents are being averaged instead of seven. The gain from
similarity does not make up for the loss from pooling fewer units.

**Similarity is only worth buying when it buys a better assumption.** Here the
assumption is about how the rate would have *changed*, and there is no reason
a 95 officer department's rate changes more like a 54 officer department's
than like a 900 officer department's. Size predicts the level and the noise,
not the trend.

Matching earns its place when the matching variable actually drives the
counterfactual path. Agency type would be a better candidate here than size,
because the campus agency genuinely moves differently.

</details>

---

**Next:** [Module 4: Building a Comparison Group](Module_04_Building_A_Comparison_Group.ipynb).

*Part of the Public Safety Statistics Tutorials, developed for the Washington
Data Exchange for Public Safety (WADEPS) through CISER at Washington State
University. Questions or corrections: yin.zhang@wsu.edu*